In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Import Dataset

In [31]:
df = pd.read_csv('../../data/raw/ai4i2020.csv')

### Processing Data

In [28]:
import os
import lightgbm as lgb

In [32]:
df.columns = df.columns.str.replace(r'[\[\]]', '', regex=True)
df.columns = df.columns.str.replace(' ', '_')  

In [33]:
df.head()

,UDI,Product_ID,Type,Air_temperature_K,Process_temperature_K,Rotational_speed_rpm,Torque_Nm,Tool_wear_min,Machine_failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [34]:
df['Type'] = df['Type'].map({'L': 1, 'M': 2, 'H': 3})
df['Temp_Diff'] = df['Process_temperature_K'] - df['Air_temperature_K']
df['Power'] = df['Torque_Nm'] * ((df['Rotational_speed_rpm'] * 2 * np.pi) / 60.0)
df['OSF_Load'] = df['Tool_wear_min'] * df['Torque_Nm']

In [35]:
feature_cols = [
    'Type',
    'Air_temperature_K', 
    'Process_temperature_K', 
    'Rotational_speed_rpm',
    'Torque_Nm',
    'Tool_wear_min',
    'Temp_Diff',
    'Power',
    'OSF_Load'
]

In [37]:
X = df[feature_cols]
y = df['Machine_failure']

### Split Data

In [38]:
from sklearn.model_selection import train_test_split

In [39]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [40]:
ratio = (len(y_train) - sum(y_train)) / sum(y_train)

In [41]:
train_data = lgb.Dataset(X_train, label=y_train)
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'scale_pos_weight': ratio,
    'learning_rate': 0.05,
    'verbose': -1
}
model = lgb.train(params, train_data, num_boost_round=100)

In [43]:
model.save_model("model/lgb_model.txt")

### Predict Data

In [49]:
from sklearn.metrics import classification_report

In [44]:
y_pred = model.predict(X_test)

In [45]:
y_pred_labels = (y_pred >= 0.5).astype(int)

In [48]:
print(classification_report(y_test, y_pred_labels, target_names=['Normal (0)', 'Error (1)']))

              precision    recall  f1-score   support

  Normal (0)       0.99      0.99      0.99      1939
   Error (1)       0.75      0.82      0.78        61

    accuracy                           0.99      2000
   macro avg       0.87      0.91      0.89      2000
weighted avg       0.99      0.99      0.99      2000

